# AI Research Paper Summarizer

An interactive, notebook-based tool that helps users quickly understand the latest AI research papers from arXiv.

## Prerequisites

Before running the notebook, ensure the following are installed and configured:

- **Python 3.10+**
- **Jupyter Notebook or JupyterLab**
- **Ollama** installed and running locally
- A locally available Ollama model (e.g., `llama3.2`)
- **Playwright** with Chromium installed
- Required Python packages:
  - `playwright`
  - `openai`
  - `ipywidgets`
  - `IPython`
  - `markdown` (optional, for rendering)

### Setup

```bash
# Install Python dependencies
pip install playwright openai ipywidgets markdown

# Install Chromium for Playwright
playwright install chromium

# Start Ollama
ollama serve

# Pull the model (run once)
ollama pull llama3.2
````

## Features

* Scrapes the latest papers from the `cs.AI` category on arXiv
* Displays a numbered list of available research papers
* Allows users to select a paper of interest interactively
* Generates a structured, easy-to-read summary using a local Ollama LLM
* Produces approximately 500-word Markdown summaries covering:

  * Overview
  * Problem Statement
  * Proposed Approach
  * Key Contributions
  * Results and Findings
  * Limitations
  * Potential Applications
  * Takeaways
* Renders summaries directly within the Jupyter Notebook
* Optionally saves summaries as Markdown (`.md`) files for future reference

## Goal

Bridge the gap between complex academic research and quick comprehension by enabling users to explore and understand cutting-edge AI papers in minutes rather than hours.

In [ ]:
!playwright install chromium


In [ ]:
from pathlib import Path
from playwright.async_api import async_playwright
from IPython.display import Markdown, display
from openai import OpenAI
import ipywidgets as widgets

In [ ]:
ARXIV_URL = "https://arxiv.org/list/cs.AI/recent"
OLLAMA_BASE_URL="http://localhost:11434/v1"
MODEL = "llama3.2"

In [ ]:
SYSTEM_PROMPT = """
You are an expert AI Research Assistant and Technical Writer.

Your task is to analyze AI research papers and produce clear, accurate, and structured summaries for engineers, researchers, and students.

Guidelines:
- Focus on the paper's main ideas and contributions.
- Explain technical concepts in simple, accessible language while preserving accuracy.
- Highlight the motivation, methodology, findings, limitations, and real-world applications.
- Do not invent information that is not present in the paper.
- If certain details are unavailable, explicitly state that they were not provided.
- Avoid unnecessary jargon and overly academic phrasing.
- Write in a professional, educational tone.

Generate the output in Markdown using exactly these sections:

# Overview
A high-level description of the paper and its primary objective.

# Problem Statement
What problem does the paper aim to solve, and why is it important?

# Proposed Approach
Describe the methodology, architecture, algorithms, or techniques introduced.

# Key Contributions
Provide 3-5 concise bullet points summarizing the main contributions.

# Results and Findings
Summarize the important findings, experiments, evaluations, or observations.

# Limitations
Mention any limitations, assumptions, or open challenges discussed in the paper.

# Potential Applications
Describe practical applications and possible impact areas.

# Takeaways
Provide a concise conclusion and key insights from the paper.

Aim for approximately 500 words.
"""

In [ ]:
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

def summarize_with_ollama(title: str, content: str) -> str:
    user_prompt = f"""
    Research Paper Title:
    {title}

    Paper Content:
    {content}

    Please analyze the paper and generate the structured Markdown summary.
    """

    response = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )

    return response.choices[0].message.content

In [ ]:
async def fetch_papers():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(
            "https://arxiv.org/list/cs.AI/recent",
            wait_until="domcontentloaded"
        )

        title_elements = page.locator(".list-title")
        total = await title_elements.count()

        papers = []

        for i in range(total):
            title_element = title_elements.nth(i)

            title = (
                await title_element.text_content()
            ).replace("Title:", "").strip()

            abs_url = await title_element.evaluate("""
                element => {
                    const dd = element.closest('dd');
                    const dt = dd?.previousElementSibling;
                    const link = dt?.querySelector(
                        'a[href*="/abs/"]'
                    );

                    return link
                        ? 'https://arxiv.org' + link.getAttribute('href')
                        : null;
                }
            """)

            if abs_url:
                papers.append({
                    "title": title,
                    "url": abs_url
                })

        await browser.close()
        return papers


papers = await fetch_papers()
print(f"Found {len(papers)} papers.")

In [ ]:
paper_dropdown = widgets.Dropdown(
    options=[
        (f"{i + 1}. {paper['title']}", i)
        for i, paper in enumerate(papers)
    ],
    description="Paper:",
    layout=widgets.Layout(width="95%")
)

summarize_button = widgets.Button(
    description="Summarize",
    button_style="success"
)

output = widgets.Output()

selected_paper = None


def on_click(b):
    global selected_paper

    selected_paper = papers[paper_dropdown.value]

    with output:
        output.clear_output()
        print("Selected:")
        print(selected_paper["title"])
        print()
        print(
            "Run the next cell to generate the summary."
        )


summarize_button.on_click(on_click)

display(paper_dropdown)
display(summarize_button)
display(output)

In [ ]:
async def summarize_selected():
    print("Opening:")
    print(selected_paper["title"])
    print()

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True
        )

        page = await browser.new_page()

        await page.goto(
            selected_paper["url"],
            wait_until="domcontentloaded"
        )

        authors = (
            await page.locator(
                "div.authors"
            ).text_content()
        ).replace(
            "Authors:",
            ""
        ).strip()

        abstract = (
            await page.locator(
                "blockquote.abstract"
            ).text_content()
        ).replace(
            "Abstract:",
            ""
        ).strip()

        await browser.close()

    paper_content = f"""
    Authors:
    {authors}

    Abstract:
    {abstract}
    """

    print("Generating summary...\n")

    summary = summarize_with_ollama(
        selected_paper["title"],
        paper_content
    )

    markdown_content = f"""
    # {selected_paper['title']}

    **Authors:** {authors}

    {summary}
    """

    display(
        Markdown(markdown_content)
    )

In [ ]:
await summarize_selected()